In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, sha2

# Initialize Spark Session
spark = SparkSession.builder.appName("PharmaGoldLayer").getOrCreate()

# 1. Create Drug Dimension Table
dim_drug = spark.read.table("silver_drug") \
    .select("clean_drug_name", "drug_role", "route") \
    .distinct() \
    .withColumn("drug_key", sha2(concat_ws("||", col("clean_drug_name"), col("drug_role")), 256))

dim_drug.write.format("delta").mode("overwrite").saveAsTable("gold_dim_drug")

# 2. Create Reaction Dimension Table    
dim_reaction = spark.read.table("silver_reaction") \
    .select("reaction_term") \
    .distinct() \
    .withColumn("reaction_key", sha2(col("reaction_term"), 256))

dim_reaction.write.format("delta").mode("overwrite").saveAsTable("gold_dim_reaction")

# 3. Create Fact Table (Joining Silver tables & generating surrogate keys)
fact_adverse_event = spark.read.table("silver_demographics").alias("demo") \
    .join(spark.read.table("silver_drug").alias("drug"), "primaryid", "inner") \
    .join(spark.read.table("silver_reaction").alias("reac"), "primaryid", "inner") \
    .select(
        sha2(concat_ws("||", col("demo.primaryid"), col("drug.clean_drug_name"), col("reac.reaction_term")), 256).alias("fact_key"),
        col("demo.primaryid"),
        col("demo.caseid"),
        col("demo.event_date"),
        col("demo.patient_age"),
        col("demo.gndr_cod").alias("gender"),
        col("demo.country"),
        sha2(concat_ws("||", col("drug.clean_drug_name"), col("drug.drug_role")), 256).alias("drug_key"),
        sha2(col("reac.reaction_term"), 256).alias("reaction_key")
    )

fact_adverse_event.write.format("delta").mode("overwrite").saveAsTable("gold_fact_adverse_event")

print("Gold Layer Star Schema (Fact & Dimensions) created successfully!")